In [5]:
import re
from nltk.tokenize import word_tokenize

# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
import gc, json, csv

from string import punctuation
from collections import Counter
from time import time

from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed
import threading
import timeit

import multiprocessing as mp
from concurrent.futures import as_completed,wait

import jieba
import jieba.analyse
import jieba.posseg as pseg
from hanziconv import HanziConv
import sys
import os

#jieba.enable_paddle()
import warnings
warnings.filterwarnings("ignore")

### In this code, we extract a list of keywords from the job posting column, and then we use the list of keywords to create a dataframe to record the keywords presents. We then save the dataframe as a csv file.



# Testing Code Below !!!

In [6]:
# function uses jieba to tokenize Chinese characters and NLTK's word_tokenize function to tokenize English characters. 
# The regular expression used to remove non-Chinese and non-English characters is also modified to include English characters.
def transText2Arr(sentence):
    retexts = []
    cur_senWords = []
    # Remove non-Chinese and non-English characters
    sentence = re.sub('[^\u4e00-\u9fa5a-zA-Z]+', '', sentence)
    if len(sentence) > 0:
        # Tokenize Chinese characters using jieba
        cur_senWords.extend(jieba.lcut(sentence))
        # Tokenize English characters using NLTK
        cur_senWords.extend(word_tokenize(sentence))
        cur_senWords = list(set(cur_senWords))
        retexts.extend(cur_senWords)
    else:
        retexts.extend(cur_senWords)
    return retexts

In [ ]:
# Load the stop words file
stopwords_file = 'stopwords.txt'
with open(stopwords_file, 'r', encoding='utf-8') as f:
    stopwords = [line.strip() for line in f.readlines()]

# Load the text file to be processed
text_file = 'text.txt'
with open(text_file, 'r', encoding='utf-8') as f:
    text = f.read()

# (should be done after tokenize the text into words) Perform word segmentation and lemmatization using jieba
words = pseg.cut(text)
lemmas = []
for word, pos in words:
    # Skip stop words and non-Chinese words
    if word not in stopwords and pos == 'n':
        lemma = jieba.lemmatize(word, pos='n')
        if lemma:
            lemmas.append(lemma.strip())

# Print the lemmatized words
print(' '.join(lemmas))

In [ ]:
def getStopWords():
    with open('baidu_stopwords.txt', encoding='utf8') as file:
        line_list = file.readlines()
        stopword_list = [k.strip() for k in line_list]
        stopword_set = set(stopword_list)
        print('停顿词列表，即变量stopword_list中共有%d个元素' %len(stopword_list))
        print('停顿词集合，即变量stopword_set中共有%d个元素' %len(stopword_set))
    return stopword_list

In [2]:
#The function is called "read_restxtcsv_data" and takes a single argument "curFile", which is the file path of the CSV file to be read.

#It defines a list "colNames" that contains the names of the columns in the CSV file.

#It uses the pandas "read_csv" function to read the CSV file specified by "curFile" and store the data in a variable "resCSV". The function passes several arguments to "read_csv" to specify how the file should be read, including:

#header=None: Specifies that the CSV file has no header row.
#index_col=None: Specifies that no column should be used as the index of the DataFrame.
#names=colNames: Specifies the column names to use for the DataFrame.
#encoding='utf-8': Specifies the character encoding of the CSV file.
#quoting=csv.QUOTE_NONE: Specifies that no special quoting should be used for fields in the CSV file.
#error_bad_lines=False: Specifies that any lines in the CSV file that cannot be read should be skipped.
#sep='?': Specifies the separator used in the CSV file.
#engine='python': Specifies the parsing engine used to read the CSV file.
#Finally, the function returns the resulting DataFrame object "resCSV".

#So, to summarize, this function reads a CSV file with specific parameters, creates a pandas DataFrame object from the file data, and returns the resulting DataFrame.

def read_restxtcsv_data(curFile):
    colNames = ['招聘ID','公司ID','公司名称','城市名称','公司所在区域','工作薪酬','教育要求','工作经历',
                '工作描述','职位名称','工作名称','招聘数量','发布日期','行业名称','数据来源','大类','中类','小类','细类']
    resCSV = pd.read_csv(curFile, header=None,index_col=None, names=colNames,encoding='utf-8', quoting=csv.QUOTE_NONE, error_bad_lines=False, sep='?', engine='python')
    return resCSV

### Python function named transTextArr2Mat32 that takes three arguments: textArr, stopwords, and wv. Here is an explanation of what the code does:

- The textArr argument is expected to be a list of strings (sentences or paragraphs) in Chinese language.
- The stopwords argument is expected to be a set of Chinese stopwords, which are words that are commonly used but do not carry significant meaning (e.g., "the", "and", "a" in English).
- The wv argument is an optional argument that can be a pre-trained word embedding model in the form of gensim Word2Vec or FastText. If wv is not None, the function will only keep the words that exist in the pre-trained model, otherwise, it will keep all words.

### The function processes each sentence in textArr by doing the following:

- First, it removes all characters that are not Chinese characters using a regular expression pattern ([^\u4e00-\u9fa5]+).
- Then, it tokenizes each sentence into a list of words using the jieba.lcut function, which is a Chinese text segmentation tool.
- For each word in the sentence, it checks if it is a stopword or if its length is less than 2. If so, it is skipped. Otherwise, if wv is not None, it checks if the word exists in the pre-trained word embedding model (wv.key_to_index). If it exists, the word is added to a list called cur_senWords.
- After all words in the sentence have been processed, the function removes duplicates from cur_senWords and adds it to a list called retexts.
- Finally, the function returns a NumPy array with the contents of retexts.
Overall, this function can be used to preprocess Chinese text data for natural language processing tasks such as sentiment analysis or topic modeling.

In [ ]:
def transTextArr2Mat32(textArr,stopwords,wv=None):
    retexts = []
    for i,sentence in enumerate(textArr):
        cur_senWords = []
        sentence = str(sentence)
        sentence = re.sub('[^\u4e00-\u9fa5]+','',sentence)
        if len(sentence) > 0:
            for word in jieba.lcut(sentence):
                if  word not in stopwords and len(word)>1:
                    if wv is not None:
                        if word in wv.key_to_index:
                            cur_senWords.append(word)
                        else:
                            continue
                    else:
                        cur_senWords.append(word)
            cur_senWords = list(set(cur_senWords))
            retexts.append(cur_senWords)
        else:
            retexts.append(cur_senWords)
    return np.array(retexts,dtype=object)


### Python function that takes in three arguments:

- sDf: A pandas DataFrame containing preprocessed text data that has been converted to embeddings.
- cText: A list of embeddings representing a job posting title.
- cWv: A pre-trained word embedding model.

+ The function first applies a lambda function to the 'desc' column of the sDf DataFrame using the apply() method. This lambda function computes the cosine similarity between the cText and the embeddings in the 'desc' column using the cWv.n_similarity() method. If either cText or the embeddings in 'desc' are empty, the cosine similarity is set to 0. The resulting similarity scores are stored in sRes.

+ Finally, the function returns the index of the highest similarity score in sRes using the idxmax() method. This index corresponds to the most similar job posting title in the sDf DataFrame, which is used to infer the job classification.

In [ ]:
def getMaxIndex(sDf,cText,cWv):
    sRes = sDf['desc'].apply(lambda x: 0 if len(cText)==0 or len(x)==0 else cWv.n_similarity(cText,x))
    return sRes.idxmax()

### This function pTest(st,ed) takes two arguments st and ed, which are the starting and ending indices of a slice of data to be processed.

- The function does the following:

- matDf = transTextArr2Mat3((datDf['职位名称']+datDf['工作名称']).iloc[st:ed].values, g_stopwords, word_vectors) - This line preprocesses a slice of text data in the DataFrame datDf by concatenating the columns '职位名称' and '工作名称', then passing it to the function transTextArr2Mat3(), which returns a NumPy array of preprocessed text data.

- transDf = stDf.loc[pd.Series(matDf).apply(lambda x: getMaxIndex(stDf,x,word_vectors))].reset_index(drop=True) - This line takes the preprocessed text data matDf, applies the getMaxIndex() function with the pre-trained word_vectors to get the index of the most similar occupation description in the DataFrame stDf, and selects the corresponding row from stDf. The resulting DataFrame is stored in transDf.

- transDf = pd.concat([datDf.iloc[st:ed,:].reset_index(drop=True),transDf.iloc[:,3:7]],axis=1) - This line concatenates the slice of the original DataFrame datDf with the occupation description information in transDf.

- resStr = f"runtime: {time()-s0},shape:{transDf.shape}" - This line calculates the runtime of the function and stores the result as a string in resStr.

- return transDf - This line returns the final DataFrame transDf, which contains the original data from datDf and additional occupation description information from stDf.


In [11]:
def pTest(st,ed):
    print(f"{st}-{ed} \n")
    transDf = None
    s0 = time()       
    matDf = transTextArr2Mat3((datDf['职位名称']+datDf['工作名称']).iloc[st:ed].values, g_stopwords, word_vectors)
    transDf = stDf.loc[pd.Series(matDf).apply(lambda x: getMaxIndex(stDf,x,word_vectors))].reset_index(drop=True)
    transDf = pd.concat([datDf.iloc[st:ed,:].reset_index(drop=True),transDf.iloc[:,3:7]],axis=1)
    resStr = f"runtime: {time()-s0},shape:{transDf.shape}"
    print(resStr)
    return transDf

In [12]:
#This function defines a multiprocessing function mpool that divides a task into multiple processes to be executed in parallel. The function takes two arguments: tsize and tOut.
#The tsize argument specifies the size of the task that needs to be divided into multiple processes, and tOut argument specifies the output file name to which the result should be written.
#The function first initializes some variables such as maxProcs, nsize, and procs. Then it creates a list of column names colNames and creates an empty Pandas DataFrame tDf with these column names.
#The function then creates a ProcessPoolExecutor object with a maximum number of processes set to maxProcs. It then creates a list of tasks that need to be executed in parallel using the submit method of the ProcessPoolExecutor object.
#Each task is created with the pTest function, which is not defined in the given code. The pTest function is assumed to take two arguments, sti and edi, which represent the start and end indices of a sub-task that needs to be executed.
#The function then waits for all the tasks to complete using the as_completed method of the concurrent.futures module. For each completed task, it retrieves the result and appends it to the tDf DataFrame.
#Finally, the result is saved to a CSV file with the given filename tOut, and the tDf DataFrame is deleted. The function also prints the total runtime of the task.

def mpool(tsize,tOut):
    maxProcs = 10
    nsize = 100000
    procs = int(tsize/nsize) + 1
    stm = time()
    
    colNames = ['招聘ID']
    colNames.extend((stDf['vid'] + '_judge').values.tolist())
    colNames.extend((stDf['vid'] + '_cbow').values.tolist())
     
    tDf = pd.DataFrame(columns=colNames)
    
    with ProcessPoolExecutor(max_workers=maxProcs) as tpe:
        taskList = []
        for i in range(0,procs):
            sti = i*nsize
            edi = (i+1)*nsize if (i+1)*nsize < tsize else tsize
            obj = tpe.submit(pTest, sti, edi)
            taskList.append(obj)
        for taskItem in as_completed(taskList):
            retDf = taskItem.result()
            tDf = tDf.append(retDf,ignore_index=True)
            
    tDf.to_csv(tOut,sep='?', encoding = 'utf_8_sig', index=False, header=False)
    del tDf
    print('total run time: %.3f s'%(time()-stm))

In [ ]:
#This Python code performs the following operations:
#It reads an Excel file named "our_chinese_mapping.xlsx" into a Pandas DataFrame called stDf. The data is read without specifying any particular column as the index.
#It applies a lambda function to the 'judge' column of stDf which splits the strings in the column by comma (',') and removes any leading or trailing white space in each item of the resulting list. The resulting list is then assigned back to the 'judge' column.
#It fills any missing values in the 'cbow' column of stDf with an empty string ('').
#It applies a lambda function to the 'cbow' column of `stDf' which splits the strings in the column by comma (',') and removes any leading or trailing white space in each item of the resulting list. The resulting list is then assigned back to the 'cbow' column.
#It creates a new DataFrame called istDf which is obtained by setting the index of stDf to the 'vid' column.
#It defines two string variables, dataNameTmp and cbowOutTmp, which are used as templates to generate file names later on in the code.
#The code appears to be processing some kind of data related to job postings or resumes. The cbow column likely contains some kind of embedding or feature representation of the text in the job postings or resumes. The code is cleaning and preprocessing the data to prepare it for some downstream analysis or machine learning task.

stDf = pd.read_excel('our_chinese_mapping.xlsx', index_col=None)
stDf['judge'] = stDf['judge'].apply(lambda x: [item.strip() for item in x.split(',')])
stDf['cbow'] = stDf['cbow'].fillna("")
stDf['cbow'] = stDf['cbow'].apply(lambda x: [item.strip() for item in x.split(',')])

istDf = stDf.set_index('vid')

dataNameTmp = "mapped_job_posting/job_res_%s.csv"
cbowOutTmp = "cbow_out_res/job_res_%s.csv"

In [ ]:
#This is a Python code block that uses a for loop to iterate over a range of values from 1 (inclusive) to 2 (exclusive).

#For each value of i in this range, the code block performs the following operations:
#It creates a string curFile by formatting the string dataNameTmp with the value of i. The exact format of dataNameTmp is not provided in the code snippet, so it's unclear what the resulting string will look like.
#It creates a string curCbowOut by formatting the string cbowOutTmp with the value of i. Again, the exact format of cbowOutTmp is not provided in the code snippet.
#It prints a message indicating that curFile is being processed as a CBOW file.
#It attempts to read a text CSV file using the read_restxtcsv_data() function and the curFile string. If an exception occurs during this operation, the continue statement skips to the next iteration of the loop.
#It gets the number of rows in the resulting dataframe resDf using the shape attribute and assigns this value to totalSize.
#It calls the mpool() function with totalSize and curCbowOut as arguments.
#It deletes the resDf dataframe to free up memory using the del statement.
#It calls the gc.collect() function to perform garbage collection, which frees up memory used by objects that are no longer in use.
#It prints a message indicating the run time for processing curFile.
#Note that time()-stm is not defined in the code snippet, so it's unclear what stm refers to or what the value of time() will be.

for i in range(1,2):
    
    curFile = dataNameTmp%i
    curCbowOut = cbowOutTmp%i
    print("%s is cbow..."%curFile)
    try:
        resDf = read_restxtcsv_data(curFile)
    except:
        continue
    totalSize = resDf.shape[0]
    mpool(totalSize,curCbowOut)
    del resDf
    gc.collect()
    print('%s run time: %.3f s'%(curFile,(time()-stm)))

In [9]:
# This function cbowMatch(x,xitem) takes two arguments x and xitem. x is expected to be a list of items, and xitem is expected to be a single item.
# The function searches for the presence of each item in x in the list of items associated with xitem in a dataframe istDf. The dataframe istDf has a column called cbow, which contains lists of items associated with each value of xitem.
# For each item in x, the function checks whether it appears in the list associated with xitem in the cbow column of the istDf dataframe. If it does, the function increments a counter variable icount.
# At the end of the function, icount is returned, which represents the number of items in the list x that were found in the list associated with xitem in the cbow column of the istDf dataframe.
# Overall, this function appears to be checking for the similarity between a list of items (x) and a single item (xitem) based on the number of overlapping items between the two.

def cbowMatch(x,xitem):
    icount = 0
    judgeList = istDf.loc[xitem]['cbow']
    for listItem in x:
        if listItem in judgeList:
            icount = icount + 1
    return icount

In [10]:
# This function judgeMatch takes two arguments x and xitem. It returns a count icount of the number of elements in x that also appear in the judgeList associated with the xitem in istDf dataframe.
# To break it down:
# x: This is a list of items that will be compared to the judgeList.
# xitem: This is a key that will be used to look up the judgeList from the istDf dataframe.
# icount: This is a counter variable that will keep track of the number of matches between x and judgeList.
# judgeList: This is a list of items that will be compared to x. It is retrieved from the istDf dataframe using the xitem key.
# for listItem in x: This starts a loop over each element listItem in x.
# if listItem in judgeList: This checks if listItem is in judgeList.
# icount = icount + 1: If listItem is in judgeList, then icount is incremented by 1.
# return icount: Once the loop is finished, the function returns the value of icount.
# Overall, the function is trying to find the number of items in x that also appear in the judgeList associated with xitem. It does this by looping through each item in x, checking if it is in judgeList, and incrementing a counter if it is. The final count is returned as the output of the function.

def judgeMatch(x,xitem):
    icount = 0
    judgeList = istDf.loc[xitem]['judge']
    for listItem in x:
        if listItem in judgeList:
            icount = icount + 1
    return icount